# Temporal Consistency Assumption in Animal Videos — Pilot

## 1. Goal
Frozen DINOv2의 동일 격자 위치 특징이 동물 영역에서 non-target 영역보다 안정적인지
반증 가능한 방식으로 조사한다. 새로운 모델/TCA 구현, 학습, Retrieval 성능 비교는 하지 않는다.
**설계와 실행 결과는 다르다.** 아래 코드 셀의 출력과 run 디렉터리만 이번 실행의 근거다.

TCViT는 정렬 후 시간 상관성을 이용한다. 따라서 정렬 없는 cosine만으로 TCA를 기각하지 않는다.
이 노트북의 cosine 및 centroid 정렬은 **가정 진단용 대리 측정**이며 TCA 연산의 재현이 아니다.
논문의 인물/비인물 차이를 정적 배경 전체에 그대로 확장하는 것은 별도 가설이다.
출처: [Wu et al., AAAI 2024, TCViT](https://ojs.aaai.org/index.php/AAAI/article/view/28424),
[공식 DINOv2](https://github.com/facebookresearch/dinov2).

## 2. Research Questions & Hypotheses

| 질문 | 방향 가설 | 측정 | 판단 범위 |
|---|---|---|---|
| RQ1 | H1: target의 consistency가 non-target보다 높다 | sequence별 Δ = C_target − C_non_target, identity macro mean, 부호 분포·cluster bootstrap CI | 생성 mask에 조건부인 비교. background와 occluder를 분리하지 못함 |
| RQ2 | H2: target 이동/형태 변화가 클수록 C_target이 낮다 | sequence 내 Spearman ρ(C_target, centroid displacement/source-frame gap), ρ(C_target, aligned mask IoU loss) | 이동·silhouette 대리 지표와의 연관. 실제 pose/관절 운동 또는 인과관계가 아님 |
| RQ3 | H3: 단순 위치 보정 후에도 H1의 양의 차이가 유지된다 | 동일 source support의 raw_common/aligned_common Δ 및 paired difference | 정수 patch centroid translation에 한정. non-rigid 정렬에 대한 검증 아님 |

0은 방향 가설의 자연스러운 경계이며 임의의 성능 목표가 아니다. 95% CI는 탐색적
불확실성 요약으로 사용한다. 효과 크기의 실용적 최소 기준은 **Not available**이다.
CI가 양수 쪽에 있고 identity별 부호가 대체로 일관되면 조건부 지지, 음수 쪽이면 이 표본에서
방향 가설에 반하는 증거, 0을 걸치면 판단 보류다. '차이가 없음'은 equivalence margin 없이
확정하지 않는다. 일부 개체/정렬 조건에서만 양수면 부분 지지로 분류한다.
mask 오류·빈 영역·낮은 coverage·일정한 motion으로 측정이 안 되면 판단 불가능이다.

## 3. Dataset & Sampling

**추천하는 최소 실험 하나:** PolarBearVidID RoVF Test에서 개체당 1 clip × 8 frame.
최대 13개체/13 clip/104 frame이며 실제 수는 출력으로 확인한다. seed=20261124로 각 개체의
정렬된 clip 목록에서 무작위 1개를 선택한다. motion이나 consistency를 본 뒤 선택하지 않는다.
서로 다른 zoo/camera가 모두 포함된다고 보장하지 않으며 manifest에 조건을 기록한다.
Meerkat 폴더에는 메타데이터만 확인되었으므로 이번 실행에서 제외한다.

RoVF 변환 코드는 tracklet을 20개 균등 위치로 샘플링하고 224×224로 resize한다.
여기서는 저장 index `linspace(0,19,8,dtype=int)`를 사용한다. 원본 track_info의 start/end로
원본 frame index를 복원하고 간격을 저장한다. 반복 source frame은 제외한다.
FPS/실시간 간격은 **Not verified**이므로 frame gap을 초로 해석하지 않는다.
이것은 연속 원본 프레임 실험과 다른 시간 척도다.

현재 관측 입력은 배포된 개체 이미지에서 변환된 crop이지 원래의 전체 camera 영상이 아니다.
추적 bounding box 생성/안정화 방식, camera 고정 여부는 **Not verified**.
원본 경로가 `data/raw/`여도 uncropped raw video라는 뜻이 아니다.
원 영상의 target/background 이동과 crop 내부 이동을 구분해야 한다.

기존 `PB_masks_Test.pkl`은 공식 수동 GT가 아닌 DINOv2-base/LDA/SAM2 기반 재생성 mask다
(`experiments/reference/rovf/segmentation/gen_pb_masks.py`). DINO 특징 기반 선택과 SAM2의 시간적
전파가 consistency 비교에 편향을 줄 수 있다. 수동 GT segmentation과 occluder label은
**Not available**. mask overlay를 저장하여 오류를 검토하되 점수에 따라 clip을 바꾸지 않는다.

사전학습 누출: generic DINOv2 checkpoint를 사용하며 RoVF fine-tuned checkpoint는 사용하지 않는다.
그러나 DINOv2 pretraining image와 평가 이미지 중복은 **Not verified**이다.
mask 생성기는 Train/Val에서 LDA를 fit하고 Test의 여러 frame에 prompt를 생성한다.
이는 독립 annotation이 아니며 완전한 zero-shot 검증으로 부르지 않는다.
Test를 탐색적으로 확인한 이후의 후속 방법 선택은 별도 validation에서 해야 한다.

## 4. Setup

RoVF reference의 기존 Python 환경을 사용한다. 필수 라이브러리는 torch, transformers,
numpy, pandas, Pillow, matplotlib, h5py, huggingface_hub이다. 추가 의존성은 설치하지 않는다.
기존 프로젝트 DINOv2 pooling 코드는 patch grid를 없애고 wildlife_tools도 import하므로
RoVF와 같은 transformers backbone의 `last_hidden_state`를 직접 사용한다.
RoVF 학습·평가 코드는 수정하지 않는다. pooling baseline/CAViT/FGRA/TCViT 구현 여부를 전제하지 않는다.

모델: cached `facebook/dinov2-small`, immutable cache revision을 기록하고 local-only로 로드한다.
Input RGB 224×224 → ImageNet normalization → frozen eval/inference_mode FP32.
기존 H5의 square resize를 유지한다. random crop/augmentation은 없다.
Output `(T,257,384)`에서 CLS는 분리하고 patch는 `(T,16,16,384)`로 복원 후 채널 L2 정규화한다.
CLS cosine은 global 참고값이며 target/background 위치 측정에는 사용하지 않는다.
입력에 mask를 곱하지 않는다. mask는 영역을 분류하는 용도로만 사용한다.

실행: 저장소 루트에서 다음 명령을 실행한다. Jupyter 없이도 셀별 stdout/figure/error가
저장된 실행본을 만들며 magic/widget 기능은 지원하지 않는다.

```powershell
experiments/reference/rovf/.venv/Scripts/python.exe scripts/run_temporal_consistency_notebook.py
```

Jupyter에서 실행하려면 위 라이브러리가 있는 kernel을 선택하고 Run All 한다.
현재 reference 환경의 ipykernel은 **Not available**이다. 원본은 `notebooks/`, 실행본·JSON·CSV·PNG는
`experiments/polarbear/temporal_consistency_pilot/<UTC-time>-<random>/`에 보관한다.
`experiments/`는 Git ignore이므로 다른 머신에서는 데이터·mask·checkpoint를 따로 준비해야 한다.

### 선택된 개체의 전체 Test 시퀀스 분석 모드

기본 파일럿 설정은 유지한다. 사용자가 요청한 확장 실행은 아래 명령으로 수행한다.

```powershell
experiments/reference/rovf/.venv/Scripts/python.exe scripts/run_temporal_consistency_notebook.py --identity-count 3 --all-test-clips
```

노트북에서 직접 실행할 때는 CONFIG의 identity_count=3, clips_per_identity=None으로 설정한다.
seed=20261124로 Test에 존재하는 개체 중 3개를 균등 비복원 추출하고, 해당 개체의 **모든 Test
클립**에서 기존과 같은 8 frame을 분석한다. Train/Val이나 원본의 모든 연속 frame을 뜻하지 않는다.
표본은 기존 결과 점수로 선택하지 않는다. `sampling.json`과 manifest로 Test 클립의 누락이 없는지
검사하며 전체 Test 모드에서 고유 frame을 구성할 수 없으면 조용히 제외하지 않고 실패한다.
실행 디렉터리는 `experiments/polarbear/temporal_consistency_identity_test/<run>/`이다.
개체별 클립 수가 달라도 sequence→identity→macro 집계로 각 identity의 가중치를 같게 둔다.
3개체 bootstrap CI는 매우 불안정하여 데이터셋 전체의 불확실성 또는 확증 근거로 해석하지 않는다.
개체별 모든 sequence의 Δ와 분포를 우선 보고한다. 앞선 13개체 단일-clip 결과와 표본의 성격이 다르다.
마스크 overlay는 모든 clip의 양 끝 frame을 12 clip씩 여러 페이지로 저장한다.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, platform, subprocess, uuid, pickle, warnings
import importlib.metadata as package_metadata
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import h5py
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoModel
from huggingface_hub import snapshot_download

ROOT = globals().get('PILOT_ROOT', Path.cwd()).resolve()
while not (ROOT / 'pyproject.toml').is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open from inside the animal-re-id repository')
    ROOT = ROOT.parent
CONFIG = dict(seed=20261124, frames=8, clips_per_identity=1, identity_count=None,
              model='facebook/dinov2-small', image_size=224, batch_size=8,
              occupancy=0.9, bootstrap=2000)
CONFIG.update(globals().get('PILOT_OVERRIDES', {}))
SEED = CONFIG['seed']
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EXPERIMENT = ('temporal_consistency_identity_test' if CONFIG['clips_per_identity'] is None
              else 'temporal_consistency_pilot')
RUN_DIR = ROOT / 'experiments/polarbear' / EXPERIMENT / (
    datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:6])
RUN_DIR.mkdir(parents=True, exist_ok=False)
BASE = ROOT / 'experiments/reference/rovf'
DATA = BASE / 'Dataset/polarbears_h5files'
MASK_PATH = BASE / 'segmentation/results/PB_masks_Test.pkl'
H5_PATH = DATA / 'clips/Test/PolarBears_test.h5'
RAW = ROOT / 'data/raw/PolarBearVidID'
for path in [MASK_PATH, H5_PATH, RAW / 'track_info.csv', RAW / 'animal_db.csv']:
    if not path.is_file():
        raise FileNotFoundError(f'Missing required local input: {path}')

def rel(path):
    return Path(path).resolve().relative_to(ROOT).as_posix()

def digest(path):
    result = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()

def git(*args):
    return subprocess.check_output(['git', '-C', str(ROOT), *args], text=True).strip()

print('Run:', rel(RUN_DIR), '| device:', DEVICE)
print('This is a pilot, not a retrieval evaluation.')
print('Effective configuration:', CONFIG)


## 5. Method

`C(p,t)=dot(z[t,p],z[t+1,p])`. 동일 격자 위치를 비교하며 동일 신체 부위의 추적으로 해석하지 않는다.
각 frame의 binary mask를 14×14 평균하여 patch occupancy를 계산한다. 두 시점에서 모두
occupancy ≥0.9인 patch를 target, ≤0.1인 patch를 non-target으로 사용하고 경계 patch는 제외한다.
0.9는 경계 오염을 줄이기 위한 분석 설정이지 성공 기준이 아니다. 0.8 설정의 민감도도 저장한다.
non-target은 mask 밖의 모든 영역이며 background/다른 동물/occluder를 분리하지 못한다.

정렬은 mask centroid 차이를 가장 가까운 정수 patch shift로 반올림한다. source `(y,x)`와
destination `(y+dy,x+dx)`를 비교한다. 이미지 밖은 버리고 wrap하지 않는다.
raw_common과 aligned_common은 **동일 source patch**가 raw와 aligned 양쪽에서 같은 class로
유지되는 교집합에서 측정한다. raw 전체 영역 수치와 common 수치를 혼합하지 않는다.
coverage 감소와 선택 편향을 함께 보고한다. 단순 정렬로 비선형 변형/시점 변화를 보정하지 못한다.
정렬을 feature cosine 최대화로 선택하지 않아 인위적 점수 상승을 줄이지만 mask 생성 편향은 남는다.

Motion: mask centroid의 normalized image-diagonal 이동량/source-frame gap.
Shape change: centroid 정렬 후 두 binary mask의 `1−IoU` (경계 clipping 포함).
둘 다 pose의 대리 지표이며 camera motion/crop jitter/segmentation 오류/실제 자세 변화를 분리하지 못한다.
Optical flow와 pose 추정은 이번 필수 실험에 도입하지 않는다.

In [ ]:
tracks = pd.read_csv(RAW / 'track_info.csv')
animals = pd.read_csv(RAW / 'animal_db.csv')
rng = np.random.default_rng(SEED)
stored_indices = np.linspace(0, 19, CONFIG['frames'], dtype=int)
manifest = []
with h5py.File(H5_PATH, 'r') as handle:
    all_keys = sorted(handle.keys())
    identities = sorted({int(k.split('_')[0]) for k in all_keys})
    if CONFIG['identity_count'] is not None:
        if not 1 <= CONFIG['identity_count'] <= len(identities):
            raise ValueError('identity_count outside available identity range')
        identities = sorted(rng.choice(identities, CONFIG['identity_count'], replace=False).tolist())
    for identity in identities:
        eligible = []
        for key in all_keys:
            ident, tracklet = map(int, key.split('_'))
            if ident != identity:
                continue
            match = tracks[(tracks.id == ident) & (tracks.tracklet == tracklet)]
            if len(match) != 1:
                raise ValueError(f'Ambiguous track mapping: {key}')
            row = match.iloc[0]
            original = np.linspace(int(row.start), int(row.end), 20, dtype=int)[stored_indices]
            if len(np.unique(original)) != len(original):
                if CONFIG['clips_per_identity'] is None:
                    raise ValueError(f'Cannot sample unique frames for all-Test clip: {key}')
                continue
            eligible.append((key, row, original))
        if not eligible:
            warnings.warn(f'No eligible clip for identity {identity}')
            continue
        indices = (np.arange(len(eligible)) if CONFIG['clips_per_identity'] is None else
                   rng.choice(len(eligible), min(CONFIG['clips_per_identity'], len(eligible)), replace=False))
        for index in indices:
            key, row, original = eligible[int(index)]
            for stored, frame in zip(stored_indices, original):
                raw_path = RAW / f'{identity:03}' / f'{identity:03}C{int(row.cam):02}T{int(row.tracklet):03}F{int(frame):03}.jpg'
                if not raw_path.is_file():
                    raise FileNotFoundError(raw_path)
                manifest.append(dict(identity=identity, clip=key, stored_index=int(stored),
                                     source_frame=int(frame), camera=int(row.cam),
                                     zoo=animals.set_index('id').loc[identity, 'zoo'], raw_path=rel(raw_path)))
manifest = pd.DataFrame(manifest)
assert not manifest.empty
manifest.to_csv(RUN_DIR / 'manifest.csv', index=False)
selected_keys = sorted(manifest['clip'].unique())
expected_keys = sorted(k for k in all_keys if int(k.split('_')[0]) in identities)
if CONFIG['clips_per_identity'] is None:
    assert selected_keys == expected_keys, 'All-Test coverage must be complete'
sampling = dict(selected_identities=identities, available_test_clips=len(expected_keys),
                selected_test_clips=len(selected_keys), all_test_clips=CONFIG['clips_per_identity'] is None,
                selected_keys=selected_keys, rule='Uniform identity sampling without replacement; seed fixed before measuring features')
(RUN_DIR/'sampling.json').write_text(json.dumps(sampling, indent=2), encoding='utf-8')
print(manifest.groupby(['identity', 'zoo', 'camera', 'clip']).size().to_string())
print('Frames:', len(manifest), 'clips:', manifest['clip'].nunique(), 'test candidates:', len(all_keys))
print('Stored indices:', stored_indices.tolist())

# Trusted local pickle only; do not use an untrusted downloaded pickle here.
with open(MASK_PATH, 'rb') as handle:
    all_masks = pickle.load(handle)
masks = {}
frames = {}
with h5py.File(H5_PATH, 'r') as handle:
    for key in manifest['clip'].unique():
        array = np.asarray(all_masks[key])
        assert array.shape == (20, 224, 224) and np.isin(array, [0, 1]).all(), key
        masks[key] = array[stored_indices].astype(bool)
        frames[key] = np.asarray(handle[key])[stored_indices]
        assert frames[key].shape == (CONFIG['frames'], 224, 224, 3)
        assert frames[key].dtype == np.uint8
del all_masks
snapshot = Path(snapshot_download(CONFIG['model'], local_files_only=True))
model = AutoModel.from_pretrained(snapshot, local_files_only=True).eval().to(DEVICE)
model.requires_grad_(False)
assert int(model.config.patch_size) == 14
assert not getattr(model.config, 'num_register_tokens', 0)
mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE)[None, :, None, None]
std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE)[None, :, None, None]
features, cls_features = {}, {}
with torch.inference_mode():
    for key, rgb in frames.items():
        x = torch.from_numpy(rgb.copy()).permute(0, 3, 1, 2).to(DEVICE).float() / 255
        output = torch.cat([model(pixel_values=(batch - mean) / std).last_hidden_state
                            for batch in x.split(CONFIG['batch_size'])])
        assert output.shape[1] == 257 and torch.isfinite(output).all()
        z = F.normalize(output, dim=-1).cpu().numpy()
        features[key] = z[:, 1:].reshape(len(rgb), 16, 16, -1)
        cls_features[key] = z[:, 0]
        print('Extracted:', key, features[key].shape, flush=True)
np.savez_compressed(RUN_DIR / 'patch_features.npz', **features)
np.savez_compressed(RUN_DIR / 'cls_features.npz', **cls_features)
metadata = dict(config=CONFIG, sampling=sampling, checkpoint_revision=snapshot.name, model_config=model.config.to_dict(),
                device=DEVICE, gpu=torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None,
                python=platform.python_version(), git_commit=git('rev-parse', 'HEAD'), git_status=git('status', '--short'),
                versions={p:package_metadata.version(p) for p in ['torch','transformers','numpy','pandas','h5py','matplotlib']},
                inputs={rel(p):digest(p) for p in [MASK_PATH, H5_PATH, RAW/'track_info.csv', RAW/'animal_db.csv']},
                manifest_sha256=digest(RUN_DIR/'manifest.csv'), source_notebook_sha256=digest(ROOT/'notebooks/temporal_consistency_pilot.ipynb'),
                annotations='Generated DINOv2/LDA/SAM2 masks; not independent GT; occluder labels Not available',
                fps='Not verified', pretraining_overlap='Not verified', status='features_extracted')
(RUN_DIR/'metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
del model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
def occupancy(mask):
    return mask.reshape(16, 14, 16, 14).mean(axis=(1, 3))

def centroid(mask):
    y, x = np.where(mask)
    return np.array([y.mean(), x.mean()]) if len(y) else np.array([np.nan, np.nan])

def correspondence(shape, shift):
    y, x = np.indices(shape)
    dy, dx = map(int, shift)
    valid = (y+dy >= 0) & (y+dy < shape[0]) & (x+dx >= 0) & (x+dx < shape[1])
    return y[valid], x[valid], y[valid]+dy, x[valid]+dx

def shape_change(a, b, shift):
    y, x, yy, xx = correspondence(a.shape, shift)
    # Full canvas union penalizes clipped foreground rather than hiding it.
    translated = np.zeros_like(a)
    translated[yy, xx] = a[y, x]
    union = np.logical_or(translated, b).sum()
    return 1 - np.logical_and(translated, b).sum()/union if union else np.nan

def measure_pair(a, b, ma, mb, threshold):
    ca, cb = centroid(ma), centroid(mb)
    if not (np.isfinite(ca).all() and np.isfinite(cb).all()):
        shift = np.array([0, 0])
        aligned_available = False
    else:
        shift = np.rint((cb-ca)/14).astype(int)
        aligned_available = True
    oa, ob = occupancy(ma), occupancy(mb)
    y, x, yy, xx = correspondence(oa.shape, shift)
    result = []
    for region in ['target', 'non_target']:
        ra, rb = (oa >= threshold, ob >= threshold) if region == 'target' else (oa <= 1-threshold, ob <= 1-threshold)
        raw_support = ra & rb
        common = ra[y,x] & rb[y,x] & rb[yy,xx]
        raw = (a*b).sum(-1)[raw_support]
        raw_common = (a[y,x]*b[y,x]).sum(-1)[common]
        aligned = (a[y,x]*b[yy,xx]).sum(-1)[common]
        for mode, values in [('raw',raw),('raw_common',raw_common),('aligned_common',aligned)]:
            if mode != 'raw' and not aligned_available:
                values = np.array([])
            result.append(dict(region=region, mode=mode, n_patches=len(values),
                               eligible_source_patches=int(ra.sum()),
                               consistency=float(values.mean()) if len(values) else np.nan,
                               coverage=len(values)/max(int(ra.sum()),1)))
    motion = np.linalg.norm(cb-ca)/np.hypot(224,224) if aligned_available else np.nan
    shape = shape_change(ma, mb, np.rint(cb-ca).astype(int)) if aligned_available else np.nan
    return result, motion, shape, shift

# Numerical checks exercise the actual analysis functions; no trained model needed.
test_rng = np.random.default_rng(7)
test_z = test_rng.normal(size=(16,16,12))
test_z /= np.linalg.norm(test_z, axis=-1, keepdims=True)
test_mask = np.zeros((224,224), bool); test_mask[56:168,56:168] = True
test_rows, test_motion, test_shape, _ = measure_pair(test_z,test_z,test_mask,test_mask,0.9)
assert all(np.isclose(r['consistency'],1) for r in test_rows)
assert test_motion == 0 and test_shape == 0
y,x,yy,xx = correspondence((16,16),(0,1))
assert len(y) == 240 and xx.min() == 1 and x.max() == 14  # no wrap
empty_rows, motion, shape, _ = measure_pair(test_z,test_z,np.zeros_like(test_mask),np.zeros_like(test_mask),0.9)
assert np.isnan(motion) and np.isnan(shape)
assert all(np.isnan(r['consistency']) for r in empty_rows if r['region']=='target')
print('Sanity checks passed: identical features, no-wrap support, empty-mask handling.')

rows = []
pair_rows = []
for key in features:
    clip_manifest = manifest[manifest['clip'] == key].sort_values('stored_index')
    identity = int(clip_manifest.identity.iloc[0])
    for t in range(CONFIG['frames']-1):
        gap = int(clip_manifest.source_frame.iloc[t+1]-clip_manifest.source_frame.iloc[t])
        cls_cosine = float(np.dot(cls_features[key][t],cls_features[key][t+1]))
        for threshold in [CONFIG['occupancy'],0.8]:
            values, motion, shape, shift = measure_pair(features[key][t],features[key][t+1],masks[key][t],masks[key][t+1],threshold)
            for row in values:
                rows.append(dict(identity=identity,clip=key,pair=t,threshold=threshold,source_frame_gap=gap,
                                 motion_per_source_frame=motion/gap,shape_change=shape,**row))
            if threshold == CONFIG['occupancy']:
                pair_rows.append(dict(identity=identity,clip=key,pair=t,source_frame_gap=gap,
                                      centroid_displacement=motion,motion_per_source_frame=motion/gap,
                                      shape_change=shape,dy=int(shift[0]),dx=int(shift[1]),cls_cosine=cls_cosine,
                                      mask_area_t=float(masks[key][t].mean()),mask_area_next=float(masks[key][t+1].mean())))
measurements = pd.DataFrame(rows)
pairs = pd.DataFrame(pair_rows)
measurements.to_csv(RUN_DIR/'pair_region_metrics.csv', index=False)
pairs.to_csv(RUN_DIR/'pair_diagnostics.csv', index=False)
print('Pair-region rows:',len(measurements),'| consecutive selected-frame pairs:',len(pairs))

## 6. Evaluation Protocol

비교 변수는 region(target/non-target), 정렬(raw/common), mask purity 민감도(0.9/0.8)뿐이다.
모델/전처리/입력/시점쌍/개체 선택은 고정한다. matched comparison의 patch 수를 저장한다.
pure region이 하나라도 없으면 해당 pair의 Δ는 결측으로 유지하고 탈락 수를 보고한다.

계층: patch 평균 → 양 region이 있는 pair의 Δ → sequence 평균 → identity 평균 → identity macro mean.
sequence/identity별 관측 수가 다를 때도 각 identity에 같은 가중치를 준다.
patch/frame을 독립 표본으로 간주하지 않는다. identity를 복원추출하는 cluster bootstrap 2,000회로
95% percentile CI를 보고한다. 동일 zoo/세션의 개체 상관은 완전히 통제하지 못한다.
기본 파일럿의 개체당 1개 sequence로는 sequence 내/개체 내 다양성에 대한 CI를 식별할 수 없다.
전체 Test 모드에서는 선택된 개체의 모든 sequence 분포를 관찰하지만 표본 개체가 3개뿐이라는 한계는 남는다.
13개 이하 cluster의 CI는 불안정하며 population 결론이나 확증 검정으로 사용하지 않는다.

RQ2는 최소 3개 유효 pair와 비상수 관측치가 있을 때만 sequence 내 rank correlation을 구한다.
ρ를 identity별 평균 후 macro 평균하며 CI는 identity bootstrap이다. 7 pair의 자기상관,
frame-gap 차이와 mask 오류 때문에 이것은 pose 가설의 직접 검증이 아니다.
motion 지표를 feature matching 점수로 정의하지 않아 C와의 수학적 순환을 피한다.

시각화: 모든 selected clip의 첫/마지막 mask overlay(contact sheet), 개체별 Δ와 CI,
motion–target C scatter(frame-gap 색), 첫 clip의 raw patch cosine heatmap 및 target/non-target
분류를 나란히 저장한다. overlay의 target 누락·다른 객체 포함·경계 흔들림을 확인한다.
수동 검토가 끝나기 전 최종적인 RQ 판정은 하지 않는다. 검토자는 `mask_qc.csv`에
pass/fail/uncertain 및 이유를 남길 수 있다. 자동 mask를 GT로 승격시키지 않는다.

mask가 없으면 전체 격자 cosine/CLS 진단만 대체 분석으로 가능하며 RQ1/RQ2의 target 비교는
판단 불가능하다. 이 노트북은 mask가 없을 때 오류로 멈춰 대체 분석과 본 실험을 섞지 않는다.
독립 target/occluder mask와 raw 영상이 확보되는 경우에만 별도 확장 실험으로 분리한다.

In [ ]:
def rho(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = np.asarray(x)[valid],np.asarray(y)[valid]
    if len(x)<3 or len(np.unique(x))<2 or len(np.unique(y))<2:
        return np.nan
    return float(np.corrcoef(pd.Series(x).rank(),pd.Series(y).rank())[0,1])

def cluster_summary(values):
    values = np.asarray(values,dtype=float)
    values = values[np.isfinite(values)]
    if not len(values):
        return dict(n_identities=0,mean=None,ci95=None,positive=0,negative=0)
    rng = np.random.default_rng(SEED)
    ci = None
    if len(values)>1:
        boot = rng.choice(values,size=(CONFIG['bootstrap'],len(values)),replace=True).mean(1)
        ci = np.quantile(boot,[0.025,0.975]).tolist()
    return dict(n_identities=len(values),mean=float(values.mean()),ci95=ci,
                positive=int((values>0).sum()),negative=int((values<0).sum()))

wide = measurements.pivot(index=['identity','clip','pair','threshold','mode'],columns='region',values='consistency').reset_index()
wide['delta'] = wide.target-wide.non_target
seq_delta = wide.groupby(['identity','clip','threshold','mode']).agg(
    delta=('delta','mean'), valid_pairs=('delta','count')).reset_index()
identity_delta = seq_delta.groupby(['identity','threshold','mode']).delta.mean().reset_index()
seq_delta.to_csv(RUN_DIR/'sequence_deltas.csv',index=False)
identity_delta.to_csv(RUN_DIR/'identity_deltas.csv',index=False)
summary = {}
for (threshold,mode),group in identity_delta.groupby(['threshold','mode']):
    summary[f'delta_{threshold}_{mode}'] = cluster_summary(group.delta)
common = seq_delta.pivot(index=['identity','clip','threshold'],columns='mode',values='delta').reset_index()
common['alignment_delta_change'] = common.aligned_common-common.raw_common
for threshold,group in common.groupby('threshold'):
    summary[f'alignment_delta_change_{threshold}'] = cluster_summary(group.groupby('identity').alignment_delta_change.mean())
rq2 = []
target = measurements[(measurements.threshold == CONFIG['occupancy']) &
                      (measurements.region=='target') & (measurements['mode']=='raw')]
for (identity,key), group in target.groupby(['identity','clip']):
    rq2.append(dict(identity=int(identity),clip=key,
                    rho_motion=rho(group.motion_per_source_frame,group.consistency),
                    rho_shape=rho(group.shape_change,group.consistency)))
rq2 = pd.DataFrame(rq2)
rq2.to_csv(RUN_DIR/'sequence_motion_correlations.csv',index=False)
for metric in ['rho_motion','rho_shape']:
    summary[metric] = cluster_summary(rq2.groupby('identity')[metric].mean())
summary['coverage'] = measurements.groupby(['threshold','mode','region']).agg(
    mean_coverage=('coverage','mean'), missing_pairs=('consistency',lambda x:int(x.isna().sum())),
    total_pairs=('consistency','size')).reset_index().to_dict('records')
summary['status'] = 'exploratory_generated_mask_analysis; independent mask QC pending'
summary['rq1_occluder'] = 'Not available: no separate occluder annotation'
summary['rq2_pose'] = 'Not verified: centroid/silhouette proxies only'
summary['identity_relevance'] = 'Not evaluated: no retrieval comparison'
(RUN_DIR/'metrics.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
pd.DataFrame([dict(clip=k,status='pending',notes='') for k in features]).to_csv(RUN_DIR/'mask_qc.csv',index=False)
print(json.dumps({k:v for k,v in summary.items() if k!='coverage'},ensure_ascii=False,indent=2))
print(pd.DataFrame(summary['coverage']).to_string(index=False))

In [ ]:
# All sampled clips are visible; no selection by a favorable metric.
keys = list(features)
for page, start in enumerate(range(0,len(keys),12),1):
    page_keys=keys[start:start+12]
    fig, axes = plt.subplots(len(page_keys),2,figsize=(7,2*len(page_keys)),squeeze=False)
    for row,key in enumerate(page_keys):
        for column,t in enumerate([0,CONFIG['frames']-1]):
            rgb = frames[key][t].astype(float)/255
            overlay = rgb.copy()
            overlay[masks[key][t]] = overlay[masks[key][t]]*0.55 + np.array([0.45,0,0])
            axes[row,column].imshow(overlay)
            axes[row,column].set_title(f'{key} stored={stored_indices[t]}')
            axes[row,column].axis('off')
    fig.tight_layout()
    fig.savefig(RUN_DIR/f'mask_overlays_page{page:02}.png',dpi=100)
    plt.show()

fig, axes = plt.subplots(1,3,figsize=(15,4))
for offset,mode in enumerate(['raw','raw_common','aligned_common']):
    group = identity_delta[(identity_delta.threshold==CONFIG['occupancy']) & (identity_delta['mode']==mode)]
    axes[0].scatter(group.identity+offset*0.12,group.delta,label=mode,s=18)
axes[0].axhline(0,color='gray',lw=1)
axes[0].set(xlabel='Identity',ylabel='Target minus non-target cosine')
axes[0].legend(fontsize=7)
scatter = axes[1].scatter(target.motion_per_source_frame,target.consistency,c=target.source_frame_gap,cmap='viridis',s=12)
axes[1].set(xlabel='Centroid displacement / source-frame gap',ylabel='Target raw cosine')
fig.colorbar(scatter,ax=axes[1],label='Source-frame gap')
for i,mode in enumerate(['raw','raw_common','aligned_common']):
    value=summary[f'delta_0.9_{mode}']
    if value['ci95'] is not None:
        axes[2].plot([i,i],value['ci95'],color='black')
        axes[2].scatter(i,value['mean'])
axes[2].axhline(0,color='gray',lw=1)
axes[2].set(xticks=range(3),xticklabels=['raw','raw common','aligned common'],ylabel='Identity macro delta with 95% CI')
fig.tight_layout(); fig.savefig(RUN_DIR/'consistency_summary.png',dpi=140); plt.show()

key=keys[0]
fig,axes=plt.subplots(1,3,figsize=(11,3))
axes[0].imshow(frames[key][0]); axes[0].set_title(f'{key} first frame')
cosine=(features[key][0]*features[key][1]).sum(-1)
im=axes[1].imshow(cosine,vmin=-1,vmax=1,cmap='coolwarm'); axes[1].set_title('Fixed-grid patch cosine')
fig.colorbar(im,ax=axes[1])
oa,ob=occupancy(masks[key][0]),occupancy(masks[key][1])
classes=np.zeros((16,16)); classes[(oa>=.9)&(ob>=.9)]=1; classes[(oa<=.1)&(ob<=.1)]=-1
axes[2].imshow(classes,vmin=-1,vmax=1,cmap='coolwarm'); axes[2].set_title('Red target / blue non-target / gray mixed')
for ax in axes: ax.axis('off')
fig.tight_layout();fig.savefig(RUN_DIR/'patch_diagnostic.png',dpi=140);plt.show()
metadata['status']='completed_exploratory_analysis_qc_pending'
(RUN_DIR/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved:',rel(RUN_DIR))
print('Final interpretation requires mask QC. Correlation is not identity relevance.')

## 7. Expected Outcomes

| 경우 | 해석 | 내릴 수 없는 결론 |
|---|---|---|
| A: target 안정성이 더 높음 | mask QC 후에도 방향이 유지되면 이 crop·encoder·시간 척도에서 조건부 지지 | identity 관련성, TCA 효과, Re-ID 향상 보장 불가 |
| B: 차이가 불명확 | CI/개체 분포가 겹치면 현재 표본으로 구별 불가; 정밀도와 영역 coverage 점검 | 가설 동등성·완전 기각 불가 |
| C: target이 더 동적 | 정적 non-target 장면에서는 단순 consistency가 배경을 선호할 가능성(추정) | 모든 animal video 또는 aligned TCA가 실패한다고 일반화 불가 |
| D: 정렬 후 변화 | raw_common 대비 aligned_common의 Δ 변화로 위치 교란 민감성 진단; 부호 반전도 기록 | patch-shift 정렬이 신체 부위 대응을 해결했다고 주장 불가 |
| E: annotation/데이터 부족 | 빈 영역·mask 실패·occluder 미분리 등으로 직접 가설 판단 불가 | 전체 crop cosine을 target-vs-background 검증으로 대체 불가 |

성공은 가정의 조건부 증거를 얻는 것이고 높은 cosine 자체가 아니다. 실패 사례와 음의 Δ,
개체별 예외, 결측/coverage도 함께 보관한다. 효과가 mask purity나 정렬 교집합 선택에
민감하면 부분 지지/판단 보류로 낮추고 해당 조건을 명시한다. 별도 occluder label이 없으므로
원 TCViT의 target-vs-non-target-person 비교는 어떤 결과에서도 직접 검증됐다고 하지 않는다.

## 8. Limitations

생성 mask의 순환성/시간 smoothing, 작은 표본, 개체별 1 clip, zoo·camera confounding,
crop 안정화 여부와 FPS 미확인, 균등 저장 frame의 가변 원본 간격, 224 square 왜곡,
patch receptive field의 배경 혼합, object motion/pose/조명 변화의 비식별성이 남는다.
DINOv2 patch token은 global attention으로 다른 영역 정보도 포함하며 pure patch도 완전한
지역 특징이 아니다. pixel displacement와 feature consistency의 차이는 물리적 움직임만의 효과가 아니다.
raw/aligned는 common support에서만 공정하게 비교하며 경계 제외율도 결과의 일부다.
독립 annotation이 없어 직접적인 RQ1 원래 형태나 occluder 분리는 판단 불가능이다.
Rank-1/Rank-3/mAP는 이번 가정 진단에 필요하지 않아 실행하지 않는다.

## 9. Minimal Implementation Plan

필수 순서는 local 입력 확인 → seed 고정 표본/원본 index manifest → generated mask load →
unmasked frozen patch 추출 → numerical sanity checks → raw/common 비교 및 motion proxies →
identity 집계/불확실성/overlay 저장 → 독립적인 mask QC → 조건부 해석이다.
실행 실패는 runner의 error output에 남긴다. 원본 notebook은 출력 없이 Git에서 관리하고
실행본과 feature cache는 run별 별도 저장한다. 기존 RoVF 코드를 수정하지 않는다.

선택 확장은 이번 run에 포함하지 않는다: 독립 GT mask가 확보된 subset 재분석,
raw 영상과 crop 비교, 독립 occluder labeling, 신체 부위 alignment, 별도 validation에서
고정된 mask purity 민감도 검증, 그 뒤 동일 split의 retrieval ablation 순서로 고려한다.
Optical flow/Pose 모델은 앞선 대리 측정으로 문제가 분리되지 않을 때만 도입한다.

## 10. Next Research Decision

QC 후 target/non-target Δ가 양수이고 개체/정렬/0.8 purity에서 방향이 유지되면 독립 mask를
확보한 소규모 validation으로 진행한다. 일부 조건에서만 유지되면 그 조건을 제한한 가설로
수정한다. Δ가 음수이면 배경 안정성에 대한 취약성을 우선 분석하고 TCA 도입을 보류한다.
정렬 후 부호가 변하면 temporal weighting 모델보다 위치 교란 분리가 먼저다.
QC 실패나 CI가 넓으면 모델을 추가하지 말고 annotation 검증과 표본의 독립성을 먼저 개선한다.

**이 최소 실험이 답할 수 있는 질문:** 선택된 crop에서 생성 mask에 조건부인 영역별 cosine
차이, centroid/silhouette 대리 변화와의 연관, 제한된 translation 보정 민감도.
**답할 수 없는 질문:** 진짜 target 대 background 대 occluder의 독립 비교, 실제 pose의 인과
효과, identity 정보의 유효성, TCA 재현/성능 향상, 다른 동물·환경으로의 일반화.

실측 결과는 위 출력과 `metrics.json`을 읽는다. mask QC 전에는 A–E 중 최종 판정을 자동으로
선언하지 않는다. 설계 문구를 실제 실험 결과로 인용하지 않는다.

In [ ]:
# Additional analysis of the completed run: no feature re-extraction.
distribution = pd.read_csv(RUN_DIR/'sequence_deltas.csv')
raw_sequences = distribution[(distribution.threshold==0.9)&(distribution['mode']=='raw')]
sequence_summary = raw_sequences.groupby('identity').delta.agg(
    clips='count', mean='mean', median='median', minimum='min', maximum='max',
    q25=lambda x:x.quantile(0.25), q75=lambda x:x.quantile(0.75),
    negative=lambda x:int((x<0).sum()))
sequence_summary.to_csv(RUN_DIR/'sequence_distribution_summary.csv')
print(sequence_summary.to_string())
identity_order=sorted(distribution.identity.unique())
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,mode in zip(axes,['raw','aligned_common']):
    selected=distribution[(distribution.threshold==0.9)&(distribution['mode']==mode)]
    values=[selected[selected.identity==identity].delta.dropna().to_numpy() for identity in identity_order]
    ax.boxplot(values,positions=np.arange(len(values)),widths=0.5)
    for position,array in enumerate(values):
        jitter=np.linspace(-0.13,0.13,len(array))
        ax.scatter(position+jitter,array,s=13,alpha=0.7)
    ax.axhline(0,color='gray',lw=1)
    ax.set(xticks=np.arange(len(values)),xticklabels=identity_order,
           xlabel='Identity',ylabel='Sequence target minus non-target cosine',title=mode)
fig.tight_layout()
fig.savefig(RUN_DIR/'sequence_distributions.png',dpi=140)
plt.show()


### 실제 실행 기록 — 설계와 구분

2026-11-24 실행: `experiments/polarbear/temporal_consistency_pilot/20261124T120206Z-d917ba/`.
13개체 × 1 clip × 8 frame = 104 frame, 91 selected-frame pair.
수치는 `metrics.json`에서 읽은 identity macro mean [95% identity-cluster bootstrap CI]다.

| 측정 (occupancy 0.9) | 실측값 |
|---|---|
| Target − Non-target, raw 전체 유효 영역 | 0.070067 [0.038559, 0.105089] |
| Target − Non-target, raw common support | 0.071497 [0.039164, 0.107228] |
| Target − Non-target, aligned common support | 0.082360 [0.048450, 0.116808] |
| Alignment에 따른 Δ 변화 (동일 support) | 0.010863 [0.002176, 0.021429] |
| Sequence 내 motion proxy와 target C의 Spearman ρ, identity 평균 | -0.230769 [-0.401099, -0.076854] |
| Sequence 내 shape proxy와 target C의 Spearman ρ, identity 평균 | -0.357143 [-0.568750, -0.131868] |

raw/aligned 모두 Δ 양수인 identity는 12/13, 음수인 identity는 1/13이다.
occupancy 0.8에서도 macro Δ의 방향이 유지된다. 모든 조건에서 영역 결측 pair는 0이다.
그러나 aligned common support 평균 coverage는 target 0.756486/non-target 0.790625로 감소한다.
이는 전체 target의 동작을 빠짐없이 관측했다는 뜻이 아니다.

**확인된 사실:** 코드 5개 셀 실행 완료, error output 0개, figure output 3개.
동일 특징 cosine=1, 경계 wrap 방지, empty mask 결측 처리 sanity check를 실행했다.
수치 CSV/JSON과 notebook schema를 검증했다. Retrieval 성능 평가는 실행하지 않았다.

**시각 검토의 범위:** 모든 13 clip의 첫/마지막 frame overlay를 살펴보았으며 일부 mask의
내부 구멍과 경계 누락을 확인했다. 중간 frame 전체 및 독립 annotation 기반 QC는
**Not verified**이다. `mask_qc.csv`는 이를 반영해 pending 상태로 유지한다.

**해석(추정):** 이 표본에서 생성 mask가 선택한 target 영역은 non-target보다 안정적이고,
큰 centroid/shape 변화와 target consistency 감소가 연관되어 있다. 독립 GT가 아닌 mask의
생성 편향 때문에 원 가설의 직접 지지로 승격하지 않는다. RQ3는 제한된 translation 보정에서
양의 차이가 유지됐다는 사실까지만 말한다. Target/background/occluder 분리 및 identity
relevance는 판단 불가능하다. 다음 단계는 TCA 모델 구현보다 독립 mask의 작은 subset 검증이다.

실행 도중 Windows cp949 console 출력 오류와 pandas `clip` 열의 attribute 충돌을 수정했다.
중단 기록은 같은 experiment 폴더의 `20261124T120006Z-0a1698`,
`20261124T120108Z-d18c03`, `20261124T120140Z-5947fa`에 남아 있다.
이 중단 시도를 별도 성공 실험으로 세지 않는다.

실행본 `experiments/polarbear/temporal_consistency_pilot/20261124T120206Z-d917ba/temporal_consistency_pilot.ipynb`,
표본 `manifest.csv`, pair별 `pair_region_metrics.csv`/`pair_diagnostics.csv`,
sequence/identity CSV, `metrics.json`/`metadata.json`, feature NPZ와 진단 PNG를 같은 run에 저장했다.
Git에는 작은 원본 notebook과 실행 도구만 포함하며 위 산출물은 복사하지 않는다.
다른 머신에서 재실행하려면 RoVF reference checkout
([Strong-AI-Lab/RoVF-Meerkat-Reidentification](https://github.com/Strong-AI-Lab/RoVF-Meerkat-Reidentification)),
변환된 H5·raw metadata·동일 생성 mask 및 cached DINOv2 checkpoint를 별도로 준비해야 한다.
자동 다운로드나 전체 평가/학습은 실행하지 않는다.

### 3개체 전체 Test 시퀀스 실행 기록 — 2026-11-24

Run: `experiments/polarbear/temporal_consistency_identity_test/20261124T120950Z-d2ad62/`.
명령: `scripts/run_temporal_consistency_notebook.py --identity-count 3 --all-test-clips`.
seed=20261124로 선택된 identity는 1, 2, 11이며 각각 Nuremberg, Berlin, Hannover에 속한다.
3개 zoo는 이번 추출 결과이지 zoo별 층화 추출을 수행한 결과가 아니다.
모든 Test 62 clip × 8 frame = 496 frame, 434 selected-frame pair를 분석했다.
Train/Val 및 원본의 모든 연속 frame을 분석한 것은 아니다. H5와 원본 track_info의 Test
배정 규칙을 별도로 대조하여 선택된 개체의 Test clip 누락이 없는지 확인했다.

| Identity | Test clips | Raw Δ mean | Median | Q25–Q75 | Negative clips |
|---|---:|---:|---:|---:|---:|
| 1 | 21 | 0.061467 | 0.052759 | 0.013097–0.100175 | 4 |
| 2 | 21 | 0.040967 | 0.035802 | -0.017329–0.086839 | 9 |
| 11 | 20 | 0.019770 | 0.021600 | -0.007137–0.052241 | 5 |


occupancy 0.9에서 identity macro Target−Non-target Δ는
raw **0.040735**, raw common **0.042298**,
aligned common **0.047053**다.
정렬의 paired Δ 변화는 **0.004755**다.
개체 내 sequence별 motion ρ의 identity macro 평균은 **-0.269473**,
shape proxy ρ는 **-0.415675**다.
3개체 bootstrap CI는 참고용으로 JSON에 남기며 데이터셋 전체에 대한 정밀한 추정으로 사용하지 않는다.

**반례와 결측:** 개체 평균은 3/3 양수지만 raw sequence Δ는 **18/62개에서 음수**다.
occupancy 0.9에서 raw target은 434 pair 중 1개, common target은 2개에서 pure patch가
없어 결측이다. 구체적으로 `001_095`의 pair=5(common), `001_096`의 pair=3(raw/common)이다.
이들을 0점으로 채우거나 clip 자체를 제외하지 않았다. pair는 0-based selected-frame pair다.
non-target 결측은 없다. threshold 0.8의 common target도 1 pair에서 결측이다.

`sequence_distributions.png`와 `sequence_distribution_summary.csv`에 전체 clip 분포를 저장했다.
마스크 overlay는 6페이지로 저장했다. 이번 확장 실행에서는 page02/page06의 양 끝 frame을
추가로 살펴보았으며 내부 구멍/target 경계 누락이 보인다. 모든 496 frame의 독립 QC는
**Not verified**이고 `mask_qc.csv`는 pending으로 유지한다.

**해석(추정):** 선택된 개체의 Test 평균은 조건부 방향 가설과 일치하지만, 클립별 음의 Δ와
생성 mask 편향 때문에 '동물 target은 항상 안정적'이라고 결론 내리지 않는다. 정렬 보정의
macro 변화가 양수여도 모든 clip에 이득이 있음을 뜻하지 않는다. Occluder 분리, 실제 pose,
identity relevance, TCA retrieval 개선은 여전히 판단할 수 없다. 다음 단계는 음의 Δ 사례와
mask 결측 사례의 독립 mask 검토다. 이 결과로 모델 추가 필요성을 확정하지 않는다.

실행 검증: 원래 5개 코드 셀은 오류 없이 완료했다. 후속 CSV 분포 분석 셀을 실제 실행해
추가했으며 모델 추론은 반복하지 않았다. Source/executed notebook 코드 일치, manifest,
feature cache 범위, CSV/JSON macro 값과 전체 Test coverage를 검사했다.
원래 feature-extraction 시점의 source hash와 후속 셀/결과 설명을 추가한 hash를 metadata에
별도로 남긴다. 대용량 산출물은 Git ignore 대상으로 유지한다.